In [7]:
training_token = 8e+9

M_min = 10
M_max = 25


param_max = training_token / M_min
param_min = training_token / M_max


param_max , param_min


(800000000.0, 320000000.0)

In [8]:
import math
import pandas as pd

# FLOP grid: 1.25e16 * 2^i for i = 0..11
flop_grid = [1.25e16 * (2 ** i) for i in range(4, 12)]

In [9]:
flop_grid

[2e+17, 4e+17, 8e+17, 1.6e+18, 3.2e+18, 6.4e+18, 1.28e+19, 2.56e+19]

In [10]:
flop_grid[0] / M_min , flop_grid[0] / M_max

(2e+16, 8000000000000000.0)

In [12]:
import math
import pandas as pd

# FLOP grid: 1.25e16 * 2^i for i = 0..11
flop_grid = [1.25e16 * (2 ** i) for i in range(4, 12)]

# Candidate model sizes from the paper-style family
# Units: parameters
paper_candidates = [
    75e6,    # 75M
    250e6,   # 250M
    500e6,   # 500M
    1e9,     # 1B
    2.5e9,   # 2.5B
    5e9,     # 5B
    10e9,    # 10B
    16e9,    # 16B
]

def human_count(x: float) -> str:
    if x >= 1e9:
        return f"{x/1e9:.3g}B"
    if x >= 1e6:
        return f"{x/1e6:.3g}M"
    if x >= 1e3:
        return f"{x/1e3:.3g}K"
    return f"{x:.0f}"

rows = []

for i, C in enumerate(flop_grid):
    # From:
    # C ~= 6ND
    # M = D/N in [10, 25]
    #
    # Since D = C / (6N),
    # M = D/N = C / (6N^2)
    #
    # Solving for N:
    # sqrt(C / (6 * 25)) <= N <= sqrt(C / (6 * 10))
    N_low = math.sqrt(C / 150.0)   # M = 25
    N_high = math.sqrt(C / 60.0)   # M = 10
    N_mid = math.sqrt(C / 120.0)   # M = 20

    D_mid = 20.0 * N_mid           # dataset size for M = 20

    valid_candidates = [
        human_count(n) for n in paper_candidates
        if N_low <= n <= N_high
    ]

    rows.append({
        "i": i,
        "FLOPs": C,
        "FLOPs_human": f"{C:.2e}",
        "N_low": N_low,
        "N_low_human": human_count(N_low),
        "N_mid_M20": N_mid,
        "N_mid_M20_human": human_count(N_mid),
        "N_high": N_high,
        "N_high_human": human_count(N_high),
        "D_mid_M20": D_mid,
        "D_mid_M20_human": human_count(D_mid),
        "valid_paper_candidates": ", ".join(valid_candidates) if valid_candidates else "none",
    })

df = pd.DataFrame(rows)

print("\n=== FLOP grid + valid model-size range ===")
print(df[[
    "i",
    "FLOPs_human",
    "N_low_human",
    "N_mid_M20_human",
    "N_high_human",
    "D_mid_M20_human",
    "valid_paper_candidates",
]].to_string(index=False))

# Optional: save to CSV
df.to_csv("scaling_candidates.csv", index=False)
print("\nSaved full table to scaling_candidates.csv")


=== FLOP grid + valid model-size range ===
 i FLOPs_human N_low_human N_mid_M20_human N_high_human D_mid_M20_human valid_paper_candidates
 0    2.00e+17       36.5M           40.8M        57.7M            816M                   none
 1    4.00e+17       51.6M           57.7M        81.6M           1.15B                    75M
 2    8.00e+17         73M           81.6M         115M           1.63B                    75M
 3    1.60e+18        103M            115M         163M           2.31B                   none
 4    3.20e+18        146M            163M         231M           3.27B                   none
 5    6.40e+18        207M            231M         327M           4.62B                   250M
 6    1.28e+19        292M            327M         462M           6.53B                   none
 7    2.56e+19        413M            462M         653M           9.24B                   500M

Saved full table to scaling_candidates.csv


In [13]:
import math
import pandas as pd

# Main fit band: use FLOP indices 4..9
# C_i = 1.25e16 * 2^i
flop_indices = list(range(4, 10))
flop_grid = [1.25e16 * (2 ** i) for i in flop_indices]

# Three candidates per FLOP
m_values = [15, 20, 25]   # M = D / N

def human_count(x: float) -> str:
    if x >= 1e9:
        return f"{x/1e9:.3f}B"
    if x >= 1e6:
        return f"{x/1e6:.3f}M"
    if x >= 1e3:
        return f"{x/1e3:.3f}K"
    return f"{x:.0f}"

rows = []

for flop_idx, C in zip(flop_indices, flop_grid):
    for M in m_values:
        # C ~= 6ND
        # M = D/N
        # => N = sqrt(C / (6M))
        # => D = M * N
        N = math.sqrt(C / (6.0 * M))
        D = M * N

        rows.append({
            "flop_idx": flop_idx,
            "flops": C,
            "flops_human": f"{C:.2e}",
            "tokens_per_param": M,
            "params": N,
            "params_human": human_count(N),
            "tokens": D,
            "tokens_human": human_count(D),
        })

df = pd.DataFrame(rows)

# Nice display order
df = df[[
    "flop_idx",
    "flops",
    "flops_human",
    "tokens_per_param",
    "params",
    "params_human",
    "tokens",
    "tokens_human",
]]

print("\n=== Exact (N, D) candidates for main 6-point fit band ===\n")
print(df[[
    "flop_idx",
    "flops_human",
    "tokens_per_param",
    "params_human",
    "tokens_human",
]].to_string(index=False))

# Save the exact numeric table
df.to_csv("main_fit_band_candidates.csv", index=False)
print("\nSaved to main_fit_band_candidates.csv")


=== Exact (N, D) candidates for main 6-point fit band ===

 flop_idx flops_human  tokens_per_param params_human tokens_human
        4    2.00e+17                15      47.140M     707.107M
        4    2.00e+17                20      40.825M     816.497M
        4    2.00e+17                25      36.515M     912.871M
        5    4.00e+17                15      66.667M       1.000B
        5    4.00e+17                20      57.735M       1.155B
        5    4.00e+17                25      51.640M       1.291B
        6    8.00e+17                15      94.281M       1.414B
        6    8.00e+17                20      81.650M       1.633B
        6    8.00e+17                25      73.030M       1.826B
        7    1.60e+18                15     133.333M       2.000B
        7    1.60e+18                20     115.470M       2.309B
        7    1.60e+18                25     103.280M       2.582B
        8    3.20e+18                15     188.562M       2.828B
        8    3.2

In [14]:
import argparse
import wandb

CASE_TO_MODEL = {
    "ratio15_L8_D512":  {"num_layers": 8,  "model_dim": 512, "num_heads": 8, "num_kv_heads": 4},
    "ratio17_L14_D384": {"num_layers": 14, "model_dim": 384, "num_heads": 6, "num_kv_heads": 3},
    "ratio20_L13_D384": {"num_layers": 13, "model_dim": 384, "num_heads": 6, "num_kv_heads": 3},
    "ratio23_L7_D512":  {"num_layers": 7,  "model_dim": 512, "num_heads": 8, "num_kv_heads": 4},
    "ratio27_L6_D512":  {"num_layers": 6,  "model_dim": 512, "num_heads": 8, "num_kv_heads": 4},
}

parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, required=True)
parser.add_argument("--case", type=str, default=None)
args, _ = parser.parse_known_args()

with wandb.init(project="parameter-golf") as run:
    case = run.config.get("case", args.case)
    if case is not None:
        overrides = CASE_TO_MODEL[case]
        # merge overrides into your OmegaConf / cfg.model here
        cfg.model.num_layers = overrides["num_layers"]
        cfg.model.model_dim = overrides["model_dim"]
        cfg.model.num_heads = overrides["num_heads"]
        cfg.model.num_kv_heads = overrides["num_kv_heads"]
        cfg.run.run_id = case

usage: ipykernel_launcher.py [-h] --config CONFIG [--case CASE]
ipykernel_launcher.py: error: the following arguments are required: --config


SystemExit: 2

c:\Users\teeds\Documents\GitHub\parameter-golf\.golf_param\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
